# Autoencoder Example 2 (PyTorch) — Denoising Autoencoder

PyTorch version of `denoising_autoencoder.ipynb`. We feed a **noisy** image and ask the
network to output the **clean** one — training pairs are `(x + noise, x)`. The bottleneck
can't carry the noise, so the model learns to redraw the underlying digit cleanly.

> Needs `torch` + `torchvision`. Trains in seconds on CPU.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)

In [ ]:
transform = transforms.ToTensor()  # [0,1]
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
NOISE = 0.5

## Model

Same dense encoder/decoder as the basic AE; the noise is added **inside the training
loop**, leaving the clean image as the target.

In [ ]:
class DenoisingAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(784, 128), nn.ReLU(),
                                     nn.Linear(128, 64), nn.ReLU(),
                                     nn.Linear(64, 32), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(32, 64), nn.ReLU(),
                                     nn.Linear(64, 128), nn.ReLU(),
                                     nn.Linear(128, 784), nn.Sigmoid())
    def forward(self, x):
        return self.decoder(self.encoder(x))

model = DenoisingAE().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## Train

`noisy = clip(clean + 0.5 * randn)`; loss compares the reconstruction of `noisy` to
`clean`.

In [ ]:
EPOCHS = 20
for epoch in range(1, EPOCHS + 1):
    model.train(); running = 0.0
    for images, _ in train_loader:
        clean = images.view(images.size(0), -1).to(device)
        noisy = torch.clamp(clean + NOISE * torch.randn_like(clean), 0.0, 1.0)
        out = model(noisy)
        loss = criterion(out, clean)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        running += loss.item() * clean.size(0)
    print(f'Epoch {epoch:2d}/{EPOCHS}  train BCE={running / len(train_dataset):.4f}')

## Noisy input → denoised output

In [ ]:
model.eval()
n = 10
clean = torch.stack([test_dataset[i][0] for i in range(n)]).view(n, -1).to(device)
noisy = torch.clamp(clean + NOISE * torch.randn_like(clean), 0.0, 1.0)
with torch.no_grad():
    denoised = model(noisy)
clean, noisy, denoised = clean.cpu().numpy(), noisy.cpu().numpy(), denoised.cpu().numpy()

rows = [(noisy, 'noisy input'), (denoised, 'denoised'), (clean, 'clean target')]
plt.figure(figsize=(n * 1.5, 4.5))
for r, (data, title) in enumerate(rows):
    for i in range(n):
        ax = plt.subplot(3, n, r * n + i + 1)
        plt.imshow(data[i].reshape(28, 28), cmap='gray'); plt.axis('off')
        if i == 0: ax.set_title(title, loc='left')
plt.show()

### Things to try

- Increase `NOISE` toward 1.0.
- Try masking pixels instead of Gaussian noise.
- Make it convolutional for sharper results.
- Reuse the encoder features for classification.